In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [ ]:
!ls -lh /content/drive/MyDrive/miniproject


total 814K
drwx------ 2 root root 4.0K Apr 14 11:00 Flicker8k_1kSubset
-rw------- 1 root root 413K Apr 14 10:05 Flickr8k_1kSubset.token.txt
-rw------- 1 root root 398K Apr 14 10:05 subset_1k_data.json


In [ ]:
!pip install git+https://github.com/fra31/auto-attack


  Cloning https://github.com/fra31/auto-attack to /tmp/pip-req-build-mve_1cjk
  Running command git clone --filter=blob:none --quiet https://github.com/fra31/auto-attack /tmp/pip-req-build-mve_1cjk
  Resolved https://github.com/fra31/auto-attack to commit a39220048b3c9f2cca9a4d3a54604793c68eca7e
  Preparing metadata (setup.py) ... done
  Created wheel for autoattack: filename=autoattack-0.1-py3-none-any.whl size=36228 sha256=c7a11b7dd129be7397906d93f821623afe9c91c25df6bd731428fc2720bca018
  Stored in directory: /tmp/pip-ephem-wheel-cache-i5wu26f7/wheels/e1/e8/28/65b2724d4c7740785979eb50bf5e1b3986ead22f6c32a87f8f
Successfully built autoattack


In [ ]:
import torch
import torchvision.transforms as transforms
from torchvision.models import resnet50
from PIL import Image
import matplotlib.pyplot as plt
from transformers import BlipProcessor, Blip2ForConditionalGeneration
from huggingface_hub import login
import gc
import os
import json
from tqdm import tqdm
from datetime import datetime
import torch.nn.functional as F
from autoattack import AutoAttack  # Ensure it's installed: pip install autoattack



In [ ]:
# --- CONFIGURATION ---
start_idx = 501        # Change for next batch: 1000, 2000, etc.
end_idx = 750
output_file = f"/content/drive/MyDrive/miniproject/blip2_adversarial_results_{start_idx}_{end_idx}.json"
epsilon = 0.5
hf_token = ""  # Replace with your HF token


In [ ]:

# Setup model and device
model_name = "Salesforce/blip2-flan-t5-xl"
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.cuda.empty_cache()
gc.collect()


30

In [ ]:
# Load the structured dataset from Google Drive
# Correct path to your file in Google Drive
dataset_path = "/content/drive/MyDrive/miniproject/subset_1k_data.json"

# Load the dataset
with open(dataset_path, "r") as f:
    dataset = json.load(f)

# Helper: clean path for Google Colab
def get_colab_path(path):
    return path.replace("C:\\Users\\swath\\OneDrive\\Documents\\miniproject\\", "/content/drive/MyDrive/miniproject/").replace("\\", "/")


In [ ]:

# Load BLIP-2
processor = BlipProcessor.from_pretrained(model_name)
model = Blip2ForConditionalGeneration.from_pretrained(
    model_name, torch_dtype=torch.float16
).to(device)


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/21.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/5.81G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.96G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

In [ ]:
# Load ResNet
resnet = resnet50(pretrained=True).to(device).eval()

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 162MB/s]


In [ ]:
# Image transforms
transform = transforms.Compose([transforms.ToTensor()])
to_pil = transforms.ToPILImage()


In [ ]:
import torch
import torch.nn.functional as F

def fgsm_attack(image, model, epsilon):
    # Make sure the image requires gradient
    image.requires_grad = True

    # Forward pass
    output = model(image)

    # Get the predicted class as the target
    pred_label = output.max(1)[1]  # pseudo-label

    # Calculate loss (you can replace with your own loss if needed)
    loss = F.cross_entropy(output, pred_label)

    # Backward pass to get gradients
    model.zero_grad()
    loss.backward()

    # Collect gradient
    gradient = image.grad.data

    # Apply FGSM perturbation
    perturbed_image = image + epsilon * gradient.sign()

    # Clamp to [0, 1]
    perturbed_image = torch.clamp(perturbed_image, 0, 1)

    return perturbed_image


In [ ]:
# Caption generator
def generate_caption(img):
    img_tensor = processor(images=img, return_tensors="pt").to(device)
    with torch.no_grad():
        with torch.cuda.amp.autocast(dtype=torch.float16):
            outputs = model.generate(**img_tensor, max_length=50)
    return processor.decode(outputs[0], skip_special_tokens=True)


In [ ]:
# PGD
def pgd_attack(image, model, epsilon=0.2, alpha=0.2, num_iters=5):
    perturbed_image = image.clone().detach().to(torch.float32)
    perturbed_image.requires_grad = True
    for _ in range(num_iters):
        output = model(perturbed_image)
        loss = F.cross_entropy(output, output.argmax(dim=1))
        model.zero_grad()
        loss.backward()
        with torch.no_grad():
            perturbed_image = perturbed_image + alpha * perturbed_image.grad.sign()
            perturbation = torch.clamp(perturbed_image - image, -epsilon, epsilon)
            perturbed_image = torch.clamp(image + perturbation, 0, 1)
        perturbed_image.requires_grad = True
    return perturbed_image


In [ ]:
# C&W
def cw_attack(image, model, targeted=False, target_class=None, c=1e-3, lr=0.05, num_iters=200):
    perturbed_image = image.clone().detach().to(device).requires_grad_(True)
    optimizer = torch.optim.Adam([perturbed_image], lr=lr)
    for _ in range(num_iters):
        output = model(perturbed_image)
        loss = -F.cross_entropy(output, target_class) if targeted else F.cross_entropy(output, output.argmax(dim=1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        perturbation = torch.clamp(perturbed_image - image, -c, c)
        perturbed_image = torch.clamp(image + perturbation, 0, 1).detach().requires_grad_(True)
    return perturbed_image


In [ ]:
# DeepFool
def deepfool_attack(image, model, num_iters=150, overshoot=0.08):
    perturbed_image = image.clone().detach().requires_grad_(True)
    for _ in range(num_iters):
        output = model(perturbed_image)
        pred_label = output.argmax(dim=1)
        loss = F.cross_entropy(output, pred_label)
        model.zero_grad()
        loss.backward()
        with torch.no_grad():
            perturbation = overshoot * perturbed_image.grad.sign()
            perturbed_image = torch.clamp(perturbed_image + perturbation, 0, 1)
        perturbed_image.requires_grad = True
    return perturbed_image


In [ ]:
# AutoAttack
def autoattack(image, model, eps=0.5):
    adversary = AutoAttack(model, norm='Linf', eps=eps, version='standard')
    return adversary.run_standard_evaluation(image, torch.tensor([model(image).argmax()], device=device), bs=1)


In [ ]:
# Set correct Google Drive image folder path
image_folder = "/content/drive/MyDrive/miniproject/Flicker8k_1kSubset"

results = []
print(f"\n⚔️ Running attacks on images {start_idx} to {end_idx}...\n")
start_time = datetime.now()
skipped = 0

for item in tqdm(dataset[start_idx:end_idx], desc="Generating captions"):
    try:
        # Correct key for image filename
        filename = item["image"]
        img_path = os.path.join(image_folder, filename)

        if not os.path.exists(img_path):
            skipped += 1
            print(f"❌ Skipped missing image: {img_path}")
            continue

        image = Image.open(img_path).convert("RGB")
        input_tensor = transform(image).unsqueeze(0).to(device).to(torch.float32).clone().detach().requires_grad_(True)

        # FGSM
        fgsm_tensor = fgsm_attack(input_tensor.clone().detach(), resnet, epsilon=0.1)

        # PGD
        pgd_tensor = pgd_attack(input_tensor.clone().detach(), resnet, epsilon=0.2, alpha=0.2, num_iters=5)

        # C&W
        cw_tensor = cw_attack(input_tensor.clone().detach(), resnet, c=1e0, lr=0.1, num_iters=200)

        # DeepFool
        df_tensor = deepfool_attack(input_tensor.clone().detach(), resnet, num_iters=50)

        # AutoAttack
        aa_tensor = autoattack(input_tensor.clone().detach(), resnet, eps=0.5)

        # Generate captions
        original_caption = generate_caption(image)
        fgsm_caption = generate_caption(to_pil(fgsm_tensor.squeeze(0).cpu()))
        pgd_caption = generate_caption(to_pil(pgd_tensor.squeeze(0).cpu()))
        cw_caption = generate_caption(to_pil(cw_tensor.squeeze(0).cpu()))
        df_caption = generate_caption(to_pil(df_tensor.squeeze(0).cpu()))
        aa_caption = generate_caption(to_pil(aa_tensor.squeeze(0).cpu()))

        # Save results
        results.append({
            "image_path": img_path,
            "ground_truth_captions": item["captions"],
            "original_caption": original_caption,
            "fgsm_caption": fgsm_caption,
            "pgd_caption": pgd_caption,
            "cw_caption": cw_caption,
            "deepfool_caption": df_caption,
            "autoattack_caption": aa_caption
        })

    except Exception as e:
        print(f"⚠️ Error processing {item.get('image', 'unknown')}: {e}")



⚔️ Running attacks on images 501 to 750...



Generating captions:   0%|          | 0/249 [00:00<?, ?it/s]

setting parameters for standard version
using standard version including apgd-ce, apgd-t, fab-t, square.
initial accuracy: 100.00%
apgd-ce - 1/1 - 1 out of 1 successfully perturbed
robust accuracy after APGD-CE: 0.00% (total time 2.7 s)
max Linf perturbation: 0.50000, nan in tensor: 0, max: 1.00000, min: 0.00000
robust accuracy: 0.00%


<ipython-input-23-2fb243935616>:5: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16):
Generating captions:   0%|          | 1/249 [00:16<1:09:40, 16.86s/it]

setting parameters for standard version
using standard version including apgd-ce, apgd-t, fab-t, square.
initial accuracy: 100.00%
apgd-ce - 1/1 - 1 out of 1 successfully perturbed
robust accuracy after APGD-CE: 0.00% (total time 2.8 s)
max Linf perturbation: 0.50000, nan in tensor: 0, max: 1.00000, min: 0.00000
robust accuracy: 0.00%


...

Generating captions:  99%|█████████▉| 247/249 [1:10:51<00:34, 17.43s/it]

setting parameters for standard version
using standard version including apgd-ce, apgd-t, fab-t, square.
initial accuracy: 100.00%
apgd-ce - 1/1 - 1 out of 1 successfully perturbed
robust accuracy after APGD-CE: 0.00% (total time 2.6 s)
max Linf perturbation: 0.50000, nan in tensor: 0, max: 1.00000, min: 0.00000
robust accuracy: 0.00%


Generating captions: 100%|█████████▉| 248/249 [1:11:08<00:17, 17.34s/it]

setting parameters for standard version
using standard version including apgd-ce, apgd-t, fab-t, square.
initial accuracy: 100.00%
apgd-ce - 1/1 - 1 out of 1 successfully perturbed
robust accuracy after APGD-CE: 0.00% (total time 2.8 s)
max Linf perturbation: 0.50000, nan in tensor: 0, max: 1.00000, min: 0.00000
robust accuracy: 0.00%


Generating captions: 100%|██████████| 249/249 [1:11:25<00:00, 17.21s/it]


In [ ]:

# Save results
with open(output_file, "w") as f:
    json.dump(results, f, indent=2)

print(f"\n✅ All done! Results saved to: {output_file}")
print(f"⏱️ Time taken: {datetime.now() - start_time}")
print(f"🚫 Skipped images: {skipped}")


✅ All done! Results saved to: /content/drive/MyDrive/miniproject/blip2_adversarial_results_501_750.json
⏱️ Time taken: 1:11:34.356252
🚫 Skipped images: 0
